### I will do transcript obtaining, text mining, and sentiment analysis of a YouTube video.

#### Install or Import Libraries

In [10]:
#install libraries
!pip install youtube-transcript-api textblob wordcloud nltk PyPDF2 numpy pandas matplotlib
!pip install beautifulsoup4 requests

In [11]:
#Import libraries for use for for transcript extraction, sentiment analysis, data organization, and visualization
from youtube_transcript_api import YouTubeTranscriptApi
from textblob import TextBlob
from wordcloud import WordCloud, STOPWORDS

import pandas as pd
import matplotlib.pyplot as plt
import re
import requests

### Choose a YouTube Video

In [12]:
#I selected a 2024 climate panelist YouTube video hosted by Harvard's Salata Institute titled "What Could Go Wrong?" 
# url: https://www.youtube.com/watch?v=d1vymVAAfR4
video_id = "d1vymVAAfR4"

print("Video ID:", video_id)

Video ID: d1vymVAAfR4


### Extract the transcript from the YouTube Video

In [13]:
def transcript_entry_to_dict(entry):
    """Attempt to convert transcript entry to a dictionary.

    """
    if isinstance(entry, dict):
        return {
            "text": entry.get("text", ""),
            "start": entry.get("start", None),
            "duration": entry.get("duration", None)
        }

    return {
        "text": getattr(entry, "text", ""),
        "start": getattr(entry, "start", None),
        "duration": getattr(entry, "duration", None)
    }


try:
    ytt_api = YouTubeTranscriptApi()
    transcript = ytt_api.fetch(video_id)
    transcript_data = [transcript_entry_to_dict(entry) for entry in transcript]

    print("Transcript successfully loaded.")
    print("Number of transcript entries:", len(transcript_data))

except Exception as e:
    transcript_data = []
    print("The transcript could not be loaded.")
    print("Possible reasons:")
    print("- The video does not have captions or a transcript.")
    print("- The transcript is disabled.")
    print("- The video ID is incorrect.")
    print("- There is a temporary connection or access issue.")
    print()
    print("Error message:")
    print(e)

Transcript successfully loaded.
Number of transcript entries: 1502


In [21]:
#Create a dataframe of the transcript
df = pd.DataFrame(transcript_data)

if df.empty:
    print("No transcript data is available. Please try another video ID.")
else:
    display(df.head())

,text,start,duration
0,welcome everyone my name is Dan,0.680,6.280
1,shrag I am a professor here in Earth and,3.359,5.561
2,planetary Sciences across the street in,6.960,3.200
3,the School of Engineering and applied,8.920,3.240
4,sciences and also at the Harvard C,10.160,5.599


### Clean the transcript text

In [22]:
def clean_text(text):
    """Remove extra spaces and line breaks."""
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


if not df.empty:
    df["clean_text"] = df["text"].apply(clean_text)
    display(df[["text", "clean_text"]].head(5))

,text,clean_text
0,welcome everyone my name is Dan,welcome everyone my name is Dan
1,shrag I am a professor here in Earth and,shrag I am a professor here in Earth and
2,planetary Sciences across the street in,planetary Sciences across the street in
3,the School of Engineering and applied,the School of Engineering and applied
4,sciences and also at the Harvard C,sciences and also at the Harvard C
5,school and I put this panel together,school and I put this panel together
6,called what could go,called what could go
7,wrong now the purpose of this panel I,wrong now the purpose of this panel I
8,want to explain just a little bit it's,want to explain just a little bit it's
9,not to wallow in terrifying things and,not to wallow in terrifying things and


## Combine the Transcript fragments into One Text

In [24]:
if not df.empty:
    full_text = " ".join(df["clean_text"].dropna().tolist())

    print("Total characters in transcript:", len(full_text))
    print()
    print(full_text[:1500])
else:
    full_text = ""
    print("No transcript text is available.")

Total characters in transcript: 54888

welcome everyone my name is Dan shrag I am a professor here in Earth and planetary Sciences across the street in the School of Engineering and applied sciences and also at the Harvard C school and I put this panel together called what could go wrong now the purpose of this panel I want to explain just a little bit it's not to wallow in terrifying things and scare everybody to death that's not the purpose although that may a side effect um I'm a climate scientist who's worked on climate change for a very long time and and the way I see the climate problem is that we're doing an experiment on the planet that hasn't been done for a very long time I my core area was in paleoclimate and so I studied earth's climate over hundreds thousands millions even billions of years and so what we're doing is extraordinary higher carbon dioxide levels and other greenhouse gases is pushing the earth into a place that hasn't been for millions of years and at a rate t

### Conduct Sentiment and Polarity Analysis Using Textblob

In [28]:
def get_polarity(text):
    return TextBlob(text).sentiment.polarity


def get_subjectivity(text):
    return TextBlob(text).sentiment.subjectivity


if not df.empty:
    df["polarity"] = df["clean_text"].apply(get_polarity)
    df["subjectivity"] = df["clean_text"].apply(get_subjectivity)

    display(df[["clean_text", "polarity", "subjectivity"]].head(10))

,clean_text,polarity,subjectivity
0,welcome everyone my name is Dan,0.8000,0.9
1,shrag I am a professor here in Earth and,0.0000,0.0
2,planetary Sciences across the street in,0.0000,0.0
3,the School of Engineering and applied,0.0000,0.0
4,sciences and also at the Harvard C,0.0000,0.0
5,school and I put this panel together,0.0000,0.0
6,called what could go,0.0000,0.0
7,wrong now the purpose of this panel I,-0.5000,0.9
8,want to explain just a little bit it's,-0.1875,0.5
9,not to wallow in terrifying things and,-1.0000,1.0


In [ ]:
# Show results in an easier to interpret format


### Interpret the Sentiment Results of the Beginning Content of the Video

#### This covers only a small introductory portion of the video's transcript, but I chose this video because it was a scientific discussion about a heated issue. Therefore, it would have a strong oscillation between neutral facts and discussion about dire consequences. The talk explains that the environmental future of the planet is very bleak. Therefore, except for the welcome introduction, all other transcript lines have a neutral or negative polarity. On subjectivity, the results are mostly zero as this is a scientific talk. However, the welcome statement and the expressed opinion about the findings have strong subjectivity. From watching the entire video, I can confirm that its entirety has little subjectivity but a strong negative polarity, due to things like the scientific near certainty of mass loss of life ahead. 